In [10]:
import pandas as pd
import numpy as np
import re
import lightgbm as lgb
import optuna

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

c:\Users\Abhi\Documents\youtube-comment-analysis\mlops-youtube-comment-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
dataset = pd.read_csv("../data/processed/processed_comments.csv")

dataset.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [ ]:
cleaned_dataset = dataset.dropna(
    subset=['clean_comment', 'category']
).copy()

X_cleaned = cleaned_dataset['clean_comment']
y_cleaned = cleaned_dataset['category']

print(cleaned_dataset.shape)
print(y_cleaned.value_counts())

In [6]:
X_train_cleaned, X_test_cleaned, y_train_cleaned, y_test_cleaned = train_test_split(
    X_cleaned,
    y_cleaned,
    test_size=0.2,
    random_state=42,
    stratify=y_cleaned
)

print("Training samples:", len(X_train_cleaned))
print("Test samples:", len(X_test_cleaned))

Training samples: 29329
Test samples: 7333


In [7]:
tfidf_cleaned = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=10000
)

X_train_tfidf_cleaned = tfidf_cleaned.fit_transform(
    X_train_cleaned
)

X_test_tfidf_cleaned = tfidf_cleaned.transform(
    X_test_cleaned
)

print("Train TF-IDF shape:", X_train_tfidf_cleaned.shape)
print("Test TF-IDF shape:", X_test_tfidf_cleaned.shape)

Train TF-IDF shape: (29329, 10000)
Test TF-IDF shape: (7333, 10000)


In [8]:
def objective(trial):

    params = {
        "objective": "multiclass",
        "num_class": 3,

        "learning_rate": trial.suggest_float(
            "learning_rate",
            1e-3,
            1e-1,
            log=True
        ),

        "n_estimators": trial.suggest_int(
            "n_estimators",
            50,
            500
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            20
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-4,
            1.0,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-4,
            1.0,
            log=True
        ),

        "class_weight": "balanced",
        "random_state": 42,
        "verbosity": -1
    }

    model = lgb.LGBMClassifier(**params)

    scores = cross_val_score(
        model,
        X_train_tfidf_cleaned,
        y_train_cleaned,
        cv=3,
        scoring="accuracy",
        n_jobs=-1
    )

    return scores.mean()

In [11]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=50
)

[I 2026-09-09 19:36:52,887] A new study created in memory with name: no-name-487805a5-e833-4460-b82e-419c4daf9f1c
[I 2026-09-09 19:37:29,580] Trial 0 finished with value: 0.6702922665902434 and parameters: {'learning_rate': 0.004264141386639574, 'n_estimators': 277, 'max_depth': 11, 'reg_alpha': 0.13169011485824175, 'reg_lambda': 0.04805780979617756}. Best is trial 0 with value: 0.6702922665902434.
[I 2026-09-09 19:38:41,997] Trial 1 finished with value: 0.8244743311861047 and parameters: {'learning_rate': 0.020317427460931254, 'n_estimators': 486, 'max_depth': 18, 'reg_alpha': 0.00011938975657405155, 'reg_lambda': 0.0003276175845268693}. Best is trial 1 with value: 0.8244743311861047.
[I 2026-09-09 19:39:27,766] Trial 2 finished with value: 0.6809302471017492 and parameters: {'learning_rate': 0.002473498057310993, 'n_estimators': 292, 'max_depth': 16, 'reg_alpha': 0.13012883636429637, 'reg_lambda': 0.008306406474666454}. Best is trial 1 with value: 0.8244743311861047.
[I 2026-09-09 19

In [12]:
best_params = study.best_params

print("Best parameters:")
print(best_params)

print("\nBest CV accuracy:")
print(study.best_value)

Best parameters:
{'learning_rate': 0.08091380297862144, 'n_estimators': 462, 'max_depth': 19, 'reg_alpha': 0.00024863581641691355, 'reg_lambda': 0.013406152001105103}

Best CV accuracy:
0.8486141738175387


In [13]:
best_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=3,
    class_weight="balanced",
    random_state=42,
    verbosity=-1,
    **best_params
)

best_model

,max_depth,19
,learning_rate,0.08091380297862144
,n_estimators,462
,objective,'multiclass'
,class_weight,'balanced'
,reg_alpha,0.00024863581641691355
,reg_lambda,0.013406152001105103
,random_state,42
,num_class,3
,verbosity,-1
,boosting_type,'gbdt'


In [14]:
best_model.fit(
    X_train_tfidf_cleaned,
    y_train_cleaned
)

,max_depth,19
,learning_rate,0.08091380297862144
,n_estimators,462
,objective,'multiclass'
,class_weight,'balanced'
,reg_alpha,0.00024863581641691355
,reg_lambda,0.013406152001105103
,random_state,42
,num_class,3
,verbosity,-1
,boosting_type,'gbdt'


In [15]:
y_train_pred = best_model.predict(
    X_train_tfidf_cleaned
)

accuracy_train = accuracy_score(
    y_train_cleaned,
    y_train_pred
)

print("Training Accuracy:", accuracy_train)

print(
    classification_report(
        y_train_cleaned,
        y_train_pred
    )
)

Training Accuracy: 0.9376385147805926
              precision    recall  f1-score   support

          -1       0.93      0.92      0.93      6598
           0       0.89      0.98      0.94     10115
           1       0.99      0.91      0.95     12616

    accuracy                           0.94     29329
   macro avg       0.94      0.94      0.94     29329
weighted avg       0.94      0.94      0.94     29329



In [16]:
y_pred = best_model.predict(
    X_test_tfidf_cleaned
)

accuracy = accuracy_score(
    y_test_cleaned,
    y_pred
)

print("Test Accuracy:", accuracy)

print(
    classification_report(
        y_test_cleaned,
        y_pred
    )
)

Test Accuracy: 0.8640392745124779
              precision    recall  f1-score   support

          -1       0.81      0.76      0.78      1650
           0       0.85      0.96      0.90      2529
           1       0.91      0.84      0.87      3154

    accuracy                           0.86      7333
   macro avg       0.85      0.85      0.85      7333
weighted avg       0.87      0.86      0.86      7333



In [17]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

def preprocess_comment(comment):

    # lowercase
    comment = comment.lower()

    # remove leading/trailing spaces
    comment = comment.strip()

    # remove newline characters
    comment = re.sub(r'\n', ' ', comment)

    # keep letters, numbers and basic punctuation
    comment = re.sub(r'[^A-Za-z0-9\s!?.,]', '', comment)

    # remove stopwords but retain sentiment-important words
    stop_words = set(stopwords.words('english')) - {
        'not', 'but', 'however', 'no', 'yet'
    }

    comment = ' '.join(
        word for word in comment.split()
        if word not in stop_words
    )

    # lemmatization
    lemmatizer = WordNetLemmatizer()

    comment = ' '.join(
        lemmatizer.lemmatize(word)
        for word in comment.split()
    )

    return comment

In [18]:
def predict_sentiment(
    comment,
    tfidf_vectorizer,
    lgbm_model
):

    cleaned_comment = preprocess_comment(comment)

    comment_tfidf = tfidf_vectorizer.transform(
        [cleaned_comment]
    )

    prediction = lgbm_model.predict(
        comment_tfidf
    )

    prediction_proba = lgbm_model.predict_proba(
        comment_tfidf
    )

    confidence = np.max(
        prediction_proba
    )

    return {
        "sentiment_class": int(prediction[0]),
        "confidence": float(confidence)
    }

In [19]:
comment = "Wow, the explanation was so clear and helpful. Definitely subscribing!"

result = predict_sentiment(
    comment,
    tfidf_cleaned,
    best_model
)

print("Predicted Sentiment:", result["sentiment_class"])
print("Confidence:", result["confidence"])

Predicted Sentiment: 1
Confidence: 0.8746034730411371
